# CI/CD Jobs
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Arrays, Sorting · **Difficulty/Frequency:** Rare (2/10)


## Concepts

**What this problem is really testing:**
- Merging intervals by sorting on start time
- The **event / sweep-line** technique — turn each interval into a "+1 starts here" and "-1 ends here" event, sort by time, then sweep through while tracking how many are currently active

**Why each one shows up here:**
- All three sub-problems are really about *when things overlap*.
- Problem 1 (merge overlapping intervals) only needs to check neighbors, once things are sorted.
- Problems 2 and 3 (where ≥2 things overlap; where the *most* things overlap) need to know the active count at every moment. Checking every single time unit is way too slow — the sweep-line trick only checks the moments where the count can actually change: every start and every end.

**The one idea to hold onto:** the number of active jobs only ever changes at a start or an end — never in between. So there's no reason to look anywhere else.

---

### Quick primers — the building blocks used below

**Sorting to enable a single pass.**
- Sorting by start time means "does this interval overlap the one before it?" becomes a question you can answer in one left-to-right scan.
- You only ever need to compare against the *most recently merged* interval — nothing earlier can matter once things are sorted.

**Event / sweep-line technique.**
- Instead of reasoning about whole intervals, break each one into two **events**: `(start, +1)` and `(end, -1)`.
- Sort all `2n` events by time, then sweep through in order, keeping a running counter: up at each `+1`, down at each `-1`.
- At any moment, that counter is exactly how many intervals are "open" right then — which is all the information you need to detect overlaps or find the busiest moment.
- **Cost:** O(n log n) to sort the `2n` events, O(n) to sweep through them — same overall cost as just sorting the intervals, but now the overlap count is exposed at every point that matters.

**Tie-breaking when two events land at the same time.**
- If a job ends at time `t` and another starts at `t`, whether that counts as "overlapping at t" depends on the problem's own definition.
- Sorting ties as `(time, -delta)` processes the `+1` before the `-1` at that same timestamp — meaning both jobs are briefly counted as active together. That matches this problem's stated rule that `(2,5)` and `(5,6)` overlap.


## Problem Statement

Given `(start, end)` job intervals:

- **Problem 1:** merge overlapping intervals into the minimal covering set. `(2,5)` and `(5,6)` count as overlapping (touching endpoints merge).
- **Problem 2:** return all sub-intervals where **>= 2** jobs run simultaneously.
- **Problem 3:** return the single interval where the **maximum** number of jobs overlap (longest such interval, if several tie for the max).

**Example** for all three: `[(2, 7), (4, 8), (15, 20)]` -> Problem 1: `[(2, 8), (15, 20)]`; Problem 2: `[(4, 7)]`; Problem 3: `(4, 7)`.


### Problem 1 -- Merge Overlapping Intervals

**Idea:** sort by start time. Walk through once, extending the last merged interval whenever the next interval's start is `<=` the last merged interval's end; otherwise start a new merged interval.

**Time complexity:** O(n log n) -- sorting dominates; the single merge pass is O(n).

**Space complexity:** O(n) for the output.


In [ ]:
from typing import List, Tuple, Optional

Interval = Tuple[int, int]


def merge_intervals(intervals: List[Interval]) -> List[Interval]:
    if not intervals:
        return []
    ordered = sorted(intervals, key=lambda x: x[0])
    merged = [list(ordered[0])]
    for start, end in ordered[1:]:
        if start <= merged[-1][1]:                 # touches or overlaps the last merged interval
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [tuple(x) for x in merged]


### Problem 2 -- Intervals With >= 2 Jobs Running

**Idea:** sweep-line. Build `(start, +1)` / `(end, -1)` events, sort by `(time, -delta)` so starts win ties. Sweep, tracking `active`; record a new overlap segment's start the instant `active` crosses from below 2 up to 2-or-more, and close it the instant `active` drops back below 2.

**Time complexity:** O(n log n) for sorting `2n` events.

**Space complexity:** O(n) for events and output.


In [ ]:
def intervals_with_two_or_more_jobs(intervals: List[Interval]) -> List[Interval]:
    events = []
    for start, end in intervals:
        events.append((start, 1))
        events.append((end, -1))
    events.sort(key=lambda e: (e[0], -e[1]))       # +1 before -1 at the same timestamp

    result = []
    active = 0
    overlap_start = None
    for time, delta in events:
        if active >= 2 and active + delta < 2:
            result.append((overlap_start, time))    # crossed back below 2 -- close the segment
            overlap_start = None
        elif active < 2 and active + delta >= 2:
            overlap_start = time                     # crossed up to >= 2 -- open a segment
        active += delta
    return result


### Problem 3 -- Busiest Interval

**Idea:** the same sweep, but track the running maximum `active` count ever seen and the (longest) contiguous stretch where the count sits at that maximum. Whenever a *new* maximum is reached, restart the candidate tracking; whenever the count drops away from the current maximum, close the candidate and keep it only if it's the longest seen so far at that maximum.

**Time complexity:** O(n log n) -- same event sort as Problem 2.

**Space complexity:** O(n) for the events list.


In [ ]:
def busiest_interval(intervals: List[Interval]) -> Optional[Interval]:
    if not intervals:
        return None
    events = []
    for start, end in intervals:
        events.append((start, 1))
        events.append((end, -1))
    events.sort(key=lambda e: (e[0], -e[1]))

    active = 0
    max_active = 0
    current_start = None
    best_start = best_end = None
    best_length = -1

    for time, delta in events:
        if delta == 1:
            active += 1
            if active > max_active:                  # a brand-new record -- restart tracking
                max_active = active
                current_start = time
                best_start, best_end, best_length = time, None, -1
            elif active == max_active:                # re-entered the current max (after a dip)
                current_start = time
        else:
            if active == max_active:                  # about to drop FROM the max -- close candidate
                length = time - current_start
                if length > best_length:
                    best_length = length
                    best_start, best_end = current_start, time
            active -= 1

    if best_end is None and best_start is not None:    # max stretch ran to the very last event
        best_end = events[-1][0]
    return (best_start, best_end)


## Verification

Run the worked example through all three, plus the tie-boundary case and edge cases the Talking Points call out.

In [ ]:
example = [(2, 7), (4, 8), (15, 20)]
assert merge_intervals(example) == [(2, 8), (15, 20)]
assert intervals_with_two_or_more_jobs(example) == [(4, 7)]
assert busiest_interval(example) == (4, 7)

# Touching endpoints count as overlapping (per the problem's explicit note)
assert merge_intervals([(2, 5), (5, 6)]) == [(2, 6)]
assert intervals_with_two_or_more_jobs([(2, 5), (5, 6)]) == [(5, 5)]   # instantaneous overlap AT t=5
assert busiest_interval([(2, 5), (5, 6)]) == (5, 5)

# Disjoint intervals: nothing overlaps
assert merge_intervals([(1, 2), (5, 6)]) == [(1, 2), (5, 6)]
assert intervals_with_two_or_more_jobs([(1, 2), (5, 6)]) == []
assert busiest_interval([(1, 2), (5, 6)]) == (1, 2)   # ties at max=1; first (and equally long) wins

# Three-way overlap
triple = [(1, 10), (2, 9), (3, 8)]
assert intervals_with_two_or_more_jobs(triple) == [(2, 9)]   # >=2 active from t=2 to t=9
assert busiest_interval(triple) == (3, 8)                     # all three active only in [3,8]

# Empty input
assert merge_intervals([]) == []
assert intervals_with_two_or_more_jobs([]) == []
assert busiest_interval([]) is None

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Intervals given as `(start, duration)`.** Convert once, up front: `(start, start + duration)`, then every function above is unchanged.
- **Streaming intervals arriving over time.** Maintain a min-heap of currently-open end times. On each new interval's start, pop any ends `<=` the new start (they've closed), then push the new end -- `len(heap)` after each update is the current active count, updated incrementally instead of re-sweeping from scratch.
- **Exactly K jobs overlapping, for arbitrary K.** Generalize Problem 2's threshold check from a hardcoded `2` to a parameter `k`, comparing `active` against `k` the same way (`active >= k` <-> `active + delta < k`, etc.).
- **In-place O(1) extra space for Problem 1.** Sort the list in place (`intervals.sort(...)` mutates rather than allocates), then merge using a write pointer into the same list instead of building a new `merged` list -- the sort itself still needs O(log n) stack space (or O(n) depending on the sort algorithm), but no separate output buffer is needed until the final trim.
- **Open vs. closed interval endpoints.** If `(2, 5)` and `(5, 6)` should NOT be considered overlapping (open intervals), switch the tie-break sort to process `-1` before `+1` at equal timestamps (so an ending job's `-1` is counted before a starting job's `+1`), which prevents the brief "both active at t" moment this problem's `<=` rule currently allows.


## Empirical complexity check

All three functions are O(n log n) (sorting dominates a linear sweep). Doubling n should scale time by a bit more than 2x (the log n factor grows slowly).

| Growth when n doubles | Implies |
|---|---|
| ~2x (or slightly more) | O(n log n) |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def make_worst_case(n):
    # n intervals, deliberately overlapping heavily (worst case for both merging
    # and the sweep, since nothing short-circuits).
    intervals = [(i, i + 50) for i in range(n)]
    return (intervals,)


solutions = {
    "merge_intervals": merge_intervals,
    "intervals_with_two_or_more_jobs": intervals_with_two_or_more_jobs,
    "busiest_interval": busiest_interval,
}
sizes = [4000, 8000, 16000, 32000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Sort by start, compare only to the last merged interval.** Once sorted, overlap-checking a new interval against everything already merged collapses to checking against just the most recent merged interval -- the sorted order guarantees nothing earlier can matter.
- **Event/sweep-line: turn intervals into `+1`/`-1` points and sweep.** Any "how many things are active at once" question over intervals reduces to this -- the active count only changes at the O(n) event points, never in between.
- **Tie-break event order encodes the overlap definition.** Whether touching endpoints count as overlapping is entirely controlled by whether `+1` or `-1` is processed first at equal timestamps -- make this choice explicit and test it directly (as the `(2,5)/(5,6)` case does above).
- **"Track a running extreme and the longest stretch at it" is a mini-pattern of its own.** Problem 3 reuses the same sweep as Problem 2 but adds "is this a new record, and how long does it last" bookkeeping -- a shape that recurs in "longest streak" and "best window" problems generally.
- **Related problems:** Merge Intervals (LeetCode 56), Meeting Rooms II (minimum rooms needed = max overlap, same sweep), Car Pooling (capacity-threshold overlap, generalizes Problem 2's `k=2` to arbitrary capacity).
- **Common pitfalls:** forgetting the event tie-break rule (silently changes whether touching intervals overlap); using `<` instead of `<=` when merging (misses the touching-endpoints case this problem explicitly requires); not deciding what to return for empty input across all three functions consistently.
